<a href="https://colab.research.google.com/github/hannaginther/ENGG680_2025_Fall/blob/main/Project/CatBoost_Clf_Regression_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 6.8 MB/s eta 0:00:00


In [7]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, cohen_kappa_score,
                             mean_squared_error, mean_absolute_error, r2_score)
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================================
# LOAD DATA
# ============================================================================
print("="*60)
print("LOADING DATA")
print("="*60)

df = pd.read_feather('/content/drive/MyDrive/Sparcs_Datafiles/model_df_clf_v2.feather')

print(f"Loaded {len(df):,} rows")
print(f" Columns: {len(df.columns)}")
print(df.head())

# ============================================================================
# 0. STRATIFIED SAMPLING (200k from 4M rows)
# ============================================================================

print("="*60)
print("CREATING STRATIFIED SAMPLING...")
print("="*60)

# Convert category to numeric
category_mapping = {'Short': 0, 'Moderate': 1, 'Long': 2, 'Extreme': 3}
if df['los_category'].dtype == 'object':
    df['los_category_numeric'] = df['los_category'].map(category_mapping)
else:
    df['los_category_numeric'] = df['los_category']

print(f"\nOriginal dataset: {len(df):,} rows")
print("\nOriginal category distribution:")
category_counts = df['los_category_numeric'].value_counts().sort_index()
print(category_counts)
print("\nProportions:")
print(df['los_category_numeric'].value_counts(normalize=True).sort_index())

# Check if we have all categories
available_categories = df['los_category_numeric'].unique()
print(f"\nAvailable categories: {sorted(available_categories)}")

# Stratified sample to maintain category distribution
SAMPLE_SIZE = 200_000

# Method 1: Use sklearn's train_split for stratified sampling
from sklearn.model_selection import train_test_split

# Calculated sample size we actually get
max_possible_sample = min(SAMPLE_SIZE, len(df))

# Use stratified sampling with sklearn
df_sample, _ = train_test_split(
    df,
    train_size=max_possible_sample,
    stratify=df['los_category_numeric'],
    random_state=42
)

print(f"\nSample dataset: {len(df_sample):,} rows")
print("\nSample category distribution:")
sample_counts = df_sample['los_category_numeric'].value_counts().sort_index()
print(sample_counts)
print("\nProportions:")
print(df_sample['los_category_numeric'].value_counts(normalize=True).sort_index())

# Verify distribution is maintained
print("\nDistribution comparison:")
print(f"{'Category':<12} {'Original %':<15} {'Sample %':<15} {'Original Count':<18} {'Sample Count':<15}")
print("-" * 85)
for cat in sorted(available_categories):
    orig_pct = (df['los_category_numeric'] == cat).mean() * 100
    sample_pct = (df_sample['los_category_numeric'] == cat).mean() * 100
    orig_count = (df['los_category_numeric'] == cat).sum()
    sample_count = (df_sample['los_category_numeric'] == cat).sum()
    print(f"{cat:<12} {orig_pct:<15.2f} {sample_pct:<15.2f} {orig_count:<18,} {sample_count:<15,}")

# Check for minimum samples per category
min_samples_per_cat = df_sample['los_category_numeric'].value_counts().min()
print(f"\nMinimum samples in any category: {min_samples_per_cat}:,")

if min_samples_per_cat < 100:
  print("WARNING: some categories have fewer than 100 samples.")
else:
  print("All categories have at least 100 samples")

# Use sampled data for rest of pipeline
df=df_sample.copy()

LOADING DATA
Loaded 4,238,636 rows
 Columns: 20
  health_service_area hospital_county facility_id age_group zip_code gender  \
0       New York City           Bronx        3058     50-69      104      F   
1       New York City           Bronx        1168     30-49      104      M   
2       New York City           Bronx        3058     50-69      104      M   
3       New York City           Bronx        1169     18-29      104      M   
4       New York City           Bronx        1169     50-69      104      F   

                     race          ethnicity  length_of_stay admission_type  \
0              Other Race   Spanish/Hispanic               1      Emergency   
1  Black/African American  Not Span/Hispanic               4      Emergency   
2              Other Race  Not Span/Hispanic               4      Emergency   
3  Black/African American  Not Span/Hispanic               5      Emergency   
4              Other Race   Spanish/Hispanic               3      Emergency   

  

In [8]:
# ============================================================================
# 1. DATA PREPARATION
# ============================================================================

# Define categorical features (all except length_of_stay and num_payment_types)
cat_features = [
    'health_service_area', 'hospital_county', 'facility_id', 'age_group',
    'zip_code', 'gender', 'race', 'ethnicity', 'admission_type',
    'ccsr_dx_code', 'ccsr_px_code', 'apr_drg_code', 'apr_mdc_code',
    'apr_severity_code', 'apr_mortality_risk', 'apr_med_surg_desc',
    'payment_type'
]

# Feature columns (exclude target variables)
feature_cols = cat_features + ['num_payment_types']

# Prepare X and y
X = df[feature_cols].copy()
y_days = df['length_of_stay'].copy()
y_category = df['los_category_numeric'].copy()

# Check distribution
print("\n" + "="*60)
print("TARGET VARIABLE DISTRIBUTION")
print("="*60)
print("\nLength of Stay Statistics: ")
print(y_days.describe())
print("\nLength of Stay Distribution: ")
print(y_category.value_counts(normalize=True).sort_index())
print(f"\nTotal samples: {len(df):,}")

# Train-test split with stratification
X_train, X_test, y_cat_train, y_cat_test, y_days_train, y_days_test = train_test_split(
    X,
    y_category,
    y_days,
    test_size=0.2,
    stratify=y_category,
    random_state=42
)

print(f"\nTraining samples: {len(X_train):,}")
print(f"Testing samples: {len(X_test):,}")


TARGET VARIABLE DISTRIBUTION

Length of Stay Statistics: 
count    200000.00000
mean          5.77725
std           8.69489
min           1.00000
25%           2.00000
50%           3.00000
75%           6.00000
max         120.00000
Name: length_of_stay, dtype: float64

Length of Stay Distribution: 
los_category_numeric
0    0.391415
1    0.317615
2    0.163970
3    0.127000
Name: proportion, dtype: float64

Total samples: 200,000

Training samples: 160,000
Testing samples: 40,000


In [16]:
# ============================================================================
# 2. STAGE 1: CLASSIFIER
# ============================================================================
print("\n" + "="*60)
print("STAGE 1: TRAINING CLASSIFIER")
print("="*60)

# Create pools
train_pool_clf = Pool(
    data=X_train,
    label=y_cat_train,
    cat_features=cat_features
)

test_pool_clf = Pool(
    data=X_test,
    label=y_cat_test,
    cat_features=cat_features
)

# Classifier with parameters optimized for imbalanced ordinal data
classifier = CatBoostClassifier(
    iterations=500,  # Reduced from 2000 for faster training
    learning_rate=0.1,  # Increased to converge faster
    depth=6,  # Reduced from 10 for speed

    # Critical for high-cardinality categorical features
    cat_features=cat_features,
    max_ctr_complexity=3,  # Reduced from 5 for speed

    # Handle class imbalance - THIS IS KEY!
    auto_class_weights='Balanced',

    # Regularization
    l2_leaf_reg=3,
    random_strength=1,

    # Sampling for better minority class learning
    bootstrap_type='Bernoulli',
    subsample=0.66,  # More aggressive subsampling for speed

    # Metric for ordinal classification
    eval_metric='WKappa',

    # Performance - using CPU
    task_type='CPU',
    thread_count=-1,

    early_stopping_rounds=50,  # Stop earlier if not improving
    verbose=50,
    random_state=42
)

# Train classifier
print("\nTraining classifier...")
classifier.fit(train_pool_clf, eval_set=test_pool_clf, plot=False)

# Evaluate classifier
y_cat_pred = classifier.predict(X_test)
y_cat_probs = classifier.predict_proba(X_test)

print("\n" + "-"*60)
print("CLASSIFIER PERFORMANCE")
print("-"*60)

kappa = cohen_kappa_score(y_cat_test, y_cat_pred, weights='quadratic')
print(f'Quadratic Weighted Kappa: {kappa:.3f}')
print(f'Accuracy: {(y_cat_test == y_cat_pred).mean():.3f}')

print("\nClassification Report:")
print(classification_report(y_cat_test, y_cat_pred,
                           target_names=['Short', 'Moderate', 'Long', 'Extreme'],
                           digits=3))

# Confusion matrix
cm = pd.crosstab(y_cat_test, y_cat_pred,
                 rownames=['Actual'], colnames=['Predicted'],
                 normalize='index')
print("\nConfusion Matrix (Row-Normalized):")
print(cm.round(3))


STAGE 1: TRAINING CLASSIFIER

Training classifier...
0:	learn: 0.6149248	test: 0.6154954	best: 0.6154954 (0)	total: 3.75s	remaining: 31m 9s
50:	learn: 0.6733359	test: 0.6784903	best: 0.6784903 (50)	total: 3m 5s	remaining: 27m 13s
100:	learn: 0.6806695	test: 0.6857450	best: 0.6857450 (100)	total: 5m 29s	remaining: 21m 42s
150:	learn: 0.6853554	test: 0.6902245	best: 0.6902245 (150)	total: 8m 53s	remaining: 20m 32s
200:	learn: 0.6884618	test: 0.6917634	best: 0.6922690 (196)	total: 12m 17s	remaining: 18m 16s
250:	learn: 0.6909941	test: 0.6932003	best: 0.6936064 (236)	total: 15m 13s	remaining: 15m 6s
300:	learn: 0.6928591	test: 0.6930833	best: 0.6937404 (281)	total: 17m 31s	remaining: 11m 35s
350:	learn: 0.6945569	test: 0.6939214	best: 0.6946123 (330)	total: 20m 6s	remaining: 8m 32s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.6946122935
bestIteration = 330

Shrink model to first 331 iterations.

------------------------------------------------------------
CLASSIFIER

ValueError: Data must be 1-dimensional, got ndarray of shape (40000, 40000) instead

In [17]:
# Evaluate classifier (FIXED VERSION)
y_cat_pred = classifier.predict(X_test)
y_cat_probs = classifier.predict_proba(X_test)

# Convert to numpy arrays to avoid pandas issues
y_cat_test_array = np.array(y_cat_test)
y_cat_pred_array = np.array(y_cat_pred).ravel()

print("\n" + "-"*60)
print("CLASSIFIER PERFORMANCE")
print("-"*60)

kappa = cohen_kappa_score(y_cat_test_array, y_cat_pred_array, weights='quadratic')
print(f'Quadratic Weighted Kappa: {kappa:.3f}')
print(f'Accuracy: {(y_cat_test_array == y_cat_pred_array).mean():.3f}')

print("\nClassification Report:")
print(classification_report(y_cat_test_array, y_cat_pred_array,
                           target_names=['Short', 'Moderate', 'Long', 'Extreme'],
                           digits=3))

# Confusion matrix
cm = pd.crosstab(y_cat_test_array, y_cat_pred_array,
                 rownames=['Actual'], colnames=['Predicted'],
                 normalize='index')
print("\nConfusion Matrix (Row-Normalized):")
print(cm.round(3))

print("\nClassifier evaluation complete. Continue to Stage 2...")


------------------------------------------------------------
CLASSIFIER PERFORMANCE
------------------------------------------------------------
Quadratic Weighted Kappa: 0.651
Accuracy: 0.544

Classification Report:
              precision    recall  f1-score   support

       Short      0.717     0.663     0.689     15656
    Moderate      0.491     0.395     0.438     12705
        Long      0.330     0.445     0.379      6559
     Extreme      0.531     0.674     0.594      5080

    accuracy                          0.544     40000
   macro avg      0.517     0.544     0.525     40000
weighted avg      0.558     0.544     0.546     40000


Confusion Matrix (Row-Normalized):
Predicted      0      1      2      3
Actual                               
0          0.663  0.218  0.090  0.028
1          0.278  0.395  0.259  0.068
2          0.073  0.220  0.445  0.261
3          0.018  0.068  0.240  0.674

Classifier evaluation complete. Continue to Stage 2...


In [ ]:
# ============================================================================
# 3. STAGE 2: CATEGORY-SPECIFIC REGRESSORS
# ============================================================================

print("\n"+"="*60)
print("STAGE 2: TRAINING CATEGORY-SPECIFIC REGRESSORS")
print("="*60)

category_regressors = {}
category_names = ['Short', 'Moderate', 'Long', 'Extreme']

for cat in range(4):
    cat_name = category_names[cat]
    print(f"\n{'-'*40}")
    print(f"Training Regressor for '{cat_name}' stays (category {cat})")
    print(f"{'-'*40}")

    # Filter data for this category
    train_mask = y_cat_train ==cat
    n_samples = train_mask.sum()

    print(f"Training samples: {n_samples:,}")

    if n_samples < 50:
      print(f"Skipping - too few samples")
      continue

      X_train_cat = X_train[train_mask]
      y_train_cat = y_days_train[train_mask]

      print(f"LOS range: {y_train_cat.min():.1f} - {y_train_cat.max():.1f} days")
      print(f"LOS mean: {y_train_cat.mean():.1f} days")

    # Adaptive hyperparameters based on category
    # Longer stays need more model capacity and less regularization!!    depth_map = {0: 5, 1: 5, 2: 6, 3: 6} # Reduced depths
    l2_map = {0: 3, 1: 3, 2: 2, 3: 2}
    iterations_map = {0: 300, 1: 400, 2: 500, 3: 500} # Reduced iterations

    regressor = CatBoostRegressor(
        iterations=iterations_map[cat],
        learning_rate=0.1,
        depth=depth_map[cat],

        loss_function='RMSE',
        eval_metric='MAE',

        cat_features=cat_features,
        l2_leaf_reg=l2_map[cat],

        task_type='CPU',
        thread_count=-1,

        early_stopping_rounds=50,
        verboose=False,
        random_state=42
        )

    # Create pool
    train_pool_reg = Pool(
        data=X_train_cat,
        label=y_train_cat,
        cat_features=cat_features
    )

    regressor.fit(train_pool_reg)
    category_regressors[cat] = regressor

    print(f"Trained successfully")

print(f"\n Trained {len(category_regressors)} category-specific regressors")

In [ ]:
# ============================================================================
# 4. PREDICTION STRATEGIES
# ============================================================================
def predict_hard_routing(X, y_cat_pred, category_regressors):
  """Route each sample to its predicted category's regressor"""
  predictions = np.zeros(len(X))

  for cat, regressor in category_regressors.items():
    mask = y_cat_pred == cat
    if mask.sum() > 0:
      predictions[mask] = regressor.predict(X[mask])

  return predictions

def predict_soft_routing(X, y_cat_probs, category_regressors):
  """Weighted ensemble using classification probabilities"""
  predictions = np.zeros(len(X))

  for cat, regressor in category_regressors.items():
    cat_preditctions = regressor.predict(X)
    predictions += cat_preditctions * y_cat_probs[:, cat]

  return predictions

# Get predictions using both strategies
print("\n" + "="*60)
print("GENERATING PREDICTIONS")
print("="*60)

y_cat_pred_test = classifier.predict(X_test)
y_cat_probs_test = classifier.predict_proba(X_test)

y_pred_hard = predict_hard_routing(X_test, y_cat_pred_test, category_regressors)
y_pred_soft = predict_soft_routing(X_test, y_cat_probs_test, category_regressors)

In [ ]:
# ============================================================================
# 5. EVALUATION
# ============================================================================

def evaluate_predictions(y_true, y_pred, y_cat_true, method_name):
    """Comprehensive evaluation of predictions"""
    print(f"\n{'='*60}")
    print(f"EVALUATION: {method_name}")
    print(f"{'='*60}")

    # Overall metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\nOverall Performance:")
    print(f"  MAE:  {mae:.2f} days")
    print(f"  RMSE: {rmse:.2f} days")
    print(f"  R²:   {r2:.3f}")

    # Performance by category
    print(f"\nPerformance by Category:")
    print(f"{'Category':<12} {'Count':<8} {'MAE':<8} {'RMSE':<8} {'R²':<8} {'Mean True':<12} {'Mean Pred':<12}")
    print("-" * 90)

    for cat in range(4):
        cat_name = category_names[cat]
        mask = y_cat_true == cat

        if mask.sum() == 0:
            continue

        mae_cat = mean_absolute_error(y_true[mask], y_pred[mask])
        rmse_cat = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
        r2_cat = r2_score(y_true[mask], y_pred[mask])
        mean_true = y_true[mask].mean()
        mean_pred = y_pred[mask].mean()

        print(f"{cat_name:<12} {mask.sum():<8} {mae_cat:<8.2f} {rmse_cat:<8.2f} "
              f"{r2_cat:<8.3f} {mean_true:<12.2f} {mean_pred:<12.2f}")

    # Performance on long stays (the critical part!)
    print(f"\nPerformance on Long Stays:")

    thresholds = [7, 14, 21, 30]
    for threshold in thresholds:
        mask = y_true > threshold
        if mask.sum() > 0:
            mae_long = mean_absolute_error(y_true[mask], y_pred[mask])
            mean_true = y_true[mask].mean()
            mean_pred = y_pred[mask].mean()
            print(f"  LOS > {threshold:2d} days (n={mask.sum():5d}): "
                  f"MAE={mae_long:6.2f}, True={mean_true:6.2f}, Pred={mean_pred:6.2f}")

    return mae, rmse, r2

# Evaluate both methods
mae_hard, rmse_hard, r2_hard = evaluate_predictions(
    y_days_test, y_pred_hard, y_cat_test, "HARD ROUTING"
)

mae_soft, rmse_soft, r2_soft = evaluate_predictions(
    y_days_test, y_pred_soft, y_cat_test, "SOFT ROUTING (RECOMMENDED)"
)

In [ ]:
# Evaluate classifier (fixed version)
y_cat_pred = classifier.predict(X_test)
y_cat_probs = classifier.predict_proba(X_test)

# Convert to numpy arrays
y_cat_test_array = np.array(y_cat_test)
y_cat_pred_array = np.array(y_cat_pred).ravel()

print("="*60)
print("CLASSIFIER PERFORMANCE")
print("="*60)

kappa = cohen_kappa_score(y_cat_test_array, y_cat_pred_array, weights='quadratic')
print(f"Quadratic Weighted Kappa: {kappa:.3f}")
print(f"Accuracy: {(y_cat_test_array == y_cat_pred_array).mean():.3f}")

print("\nClassification Report:")
print(classification_report(y_cat_test_array == y_cat_pred_array,
                            target_names=['Short', 'Moderate', 'Long', 'Extreme'],
                            digits=3))

# Confusion Matrix
cm = pd.crosstab(y_cat_test_array, y_cat_pred_array,
                 rownames=['Actual'], colnames=['Predicted'],
                 normalize='index')
print("\nConfusion Matrix (Row-Normalized): ")
print(cm.round(3))

In [ ]:
# ============================================================================
# 6. FEATURE IMPORTANCE
# ============================================================================

print("\n" + "="*60)
print("TOP FREATURES FOR CLASSIFIER")
print("="*60)

feature_importance = classifier.get_feature_importance()
feature_names = classifier.get_feature_importance(train_pool_clf)
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print(importance_df.head(30).to_string(index=False))

In [ ]:
# ============================================================================
# 7. SAVE MODELS FOR FULL DATASET TRAINING
# ============================================================================

print("\n" + "="*60)
print("SAVING MODELS")
print("="*60)

# Save models
classifier.save_model('classifier_200k.cbm')
for cat, regressor in category_regressors.items():
    regressor.save_model(f'regressor_cat{cat}_200k.cbm')

print("✓ Models saved:")
print("  - classifier_200k.cbm")
for cat in category_regressors.keys():
    print(f"  - regressor_cat{cat}_200k.cbm")

In [ ]:
print("\n" + "="*60)
print("RECOMMENDATIONS")
print("="*60)

if mae_soft < mae_hard:
  print("\nRecommended strategy: SOFT ROUTING (Probability weighted ensemble)")
  print(f"  It improves MAE by {mae_hard - mae_soft:.2f} days")
else:
  print("\nRecommended strategy: HARD ROUTING (direct category assignment)")
  print(f"  It improves MAE by {mae_soft - mae_hard:.2f} days")

if kappa < 0.65:
  print("\nClassifier could be improved: ")
  print("   - Try more iterations")
  print("   - Increase depth to 12")
  print("   - Create feature interactions")

# Check long stay performance
long_mask = y_days_test > 14
if long_mask.sum() > 0:
  mae_long = mean_absolute_error(y_days_test[long_mask], y_pred_soft[long_mask])
  if mae_long > 7:
    print("\n Long stay predictions need improvement:")
    print("   - Consider addping sample weights")
    print("   - Try quantile regression for category 3 (extreme)")
    print("   - Add feature engineering")

In [ ]:
# Load classifier
from catboost import CatBoostClassifier, CatBoostRegressor

classifier = CatBoostClassifier()
classifier.load_model('classifier_200k.cbm')

# Load regressors
category_regressors = {}
for cat in range(4):
    regressor = CatBoostRegressor()
    regressor.load_model(f'regressor_cat{cat}_200k.cbm')
    category_regressors[cat] = regressor

In [ ]:
import numpy as np
from catboost import CatBoostRegressor, Pool

# ============================================================================
# FUNCTION TO CREATE SAMPLE WEIGHTS
# ============================================================================

def create_sample_weights(y, method='log_scaled'):
  """
  Create sample weights that emphasize longer stays

  Parameters:
  -----------
  y: array-like
     Length of stay values